# Level 3 — 불균형 대응 및 고급 Augmentation

**목표**: 다수 클래스의 정확도를 크게 희생하지 않으면서, 소수 클래스 (foggy / snowy / dawn-dusk) 의 성능을 끌어올립니다.

다음 축에서 **최소 2가지 이상** 의 기법을 적용하세요.
- Loss-level: Weighted CE, Focal Loss, LDAM, Class-Balanced Loss
- Sampling-level: class-balanced sampler
- Augmentation-level: RandAugment, Mixup, CutMix

Level 1 / 2 에서 가장 좋았던 백본을 base 로 사용하세요. wandb 를 사용하면 여러 기법의 비교 Run 을 같은 프로젝트에 모아 볼 수 있어 편리합니다.

In [1]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/jjay321-oss/2026-HYU-AUE8088-PA2

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

Cloning into '2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 74 (delta 25), reused 7 (delta 7), pack-reused 33 (from 2)
Receiving objects: 100% (74/74), 90.43 KiB | 1.43 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/2026-HYU-AUE8088-PA2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.datasets.samplers import class_balanced_sampler
from src.losses.imbalanced import FocalLoss, ClassBalancedLoss, LDAMLoss, weighted_cross_entropy
from src.augment.mix import mixup_data, cutmix_data, mixed_loss
from src.models.resnet import resnet18

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level3"]
# 각 실험마다 RUN_NAME 만 바꿔서 동일 프로젝트에 누적하세요.
EXPERIMENT_NAME = "baseline-ce"#"focal+sampler"

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


In [4]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

# 옵션 A — 가장 불균형이 심한 weather 속성 기준 class-balanced sampler 사용
#sampler = class_balanced_sampler(train_ds, attribute="weather")
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=bedc7de8-01d3-49f9-9964-2f0eb4ed2985
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:01<00:00, 146MB/s]


압축 해제 중...
완료 → ../data/set_a


In [7]:
# 옵션 B — 속성별로 다른 loss 적용. 가장 불균형이 심한 속성에 가장 강한 loss 사용.
samples_w = train_ds.class_counts("weather")
samples_s = train_ds.class_counts("scene")
samples_t = train_ds.class_counts("timeofday")

loss_fns = {
    "weather":   nn.CrossEntropyLoss(),#FocalLoss(gamma=2.0).to(device),
    "scene":     nn.CrossEntropyLoss(),#ClassBalancedLoss(samples_s).to(device),
    "timeofday": nn.CrossEntropyLoss(),
}

model = resnet18().to(device)
epochs = 30
optim  = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level3-{EXPERIMENT_NAME}",
    config={
        "backbone": "resnet18",
        "sampler": "class_balanced(weather)",
        "loss": {"weather": "focal_g2.0", "scene": "cb_loss", "timeofday": "ce"},
        "epochs": epochs, "batch": BATCH, "lr": 3e-4, "seed": SEED,
    },
    tags=WANDB_TAGS + [EXPERIMENT_NAME],
)
trainer = MultiTaskTrainer(model, optim, sched, loss_fns, device, TrainConfig(epochs=epochs), logger=logger)

trainer.fit(train_loader, val_loader)

In [ ]:
trainer.fit(train_loader, val_loader)# 옵션 C — 학습 루프에 Mixup/CutMix 를 통합하여 적용
# (깨끗한 실험을 위해서는 _train_one_epoch 를 서브클래싱하는 것이 좋으나,
#  아래는 augmented step 의 핵심만 인라인으로 보인 것입니다.)

from tqdm import tqdm

def step_with_mix(images, targets):
    """50% 확률로 Mixup, 나머지 50% 확률로 CutMix 적용."""
    if torch.rand(1).item() < 0.5:
        x, ya, yb, lam = mixup_data(images, targets, alpha=0.2)
    else:
        x, ya, yb, lam = cutmix_data(images, targets, alpha=1.0)
    logits = model(x)
    return mixed_loss(loss_fns, logits, ya, yb, lam)

# TODO: step_with_mix 와 trainer.evaluate() 를 사용하여 학습 루프를 작성하세요.
# 직접 작성한 학습 루프 안에서도 logger.log({...}, step=epoch) 로 매 epoch 메트릭을 wandb 에 보낼 수 있습니다.

In [ ]:
# 학습 종료 후 — 속성별 confusion matrix + per-class F1 표를 wandb 에 업로드
val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(val_pred, val_tgt)
prf = per_class_prf(val_pred, val_tgt)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    rows = list(zip(prf[a]["class"], prf[a]["precision"], prf[a]["recall"], prf[a]["f1"], prf[a]["support"]))
    logger.log_table(f"final/prf_{a}", ["class", "P", "R", "F1", "support"], [list(r) for r in rows])
logger.finish()

os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/level3_focal_weather_sampler.pth")

## 분석 (필수)

각 기법에 대해 **속성별 per-class F1 표** 를 작성하세요. 다음 항목을 강조해 주세요.
- 소수 클래스 (foggy / snowy / dawn-dusk) 의 적용 전후 성능 차이.
- 다수 클래스의 회귀 (regression) 발생 여부 — 그 trade-off 가 정당한지 논거.
- Sampling 과 Mixup / CutMix 의 상호작용 — 서로 도움이 되는지 충돌하는지.